In [ ]:
import os
import re
import json
import argparse
from collections import Counter
from transformers import AutoTokenizer
from tqdm import tqdm


DEFAULT_INPUT_JSONL = os.environ.get("INPUT_JSONL", "/path/to/input.jsonl")
DEFAULT_MODEL_PATH = os.environ.get("MODEL_PATH", "/path/to/model")

DEFAULT_FIELD = "solution"

DEFAULT_NGRAM_N = 50
DEFAULT_THRESHOLD = 5
DEFAULT_BATCH_SIZE = 1000

    # New: default output file
DEFAULT_SAVE_IDS = "ngram.txt"

_FENCE_RE = re.compile(r"```(?:latex)?\s*(.*?)\s*```", re.DOTALL | re.IGNORECASE)


def load_tokenizer(model_path: str):
    print(f"Loading tokenizer: {model_path}")
    return AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
        use_fast=True,
    )


def extract_latex(text: str) -> str:
    """Optional: remove ```latex fence, keep only body, avoid fence interfering with repetition detection."""
    if text is None:
        return ""
    text = str(text).strip()
    m = _FENCE_RE.search(text)
    if m:
        return m.group(1).strip()
    return text


def has_repetition_ids(input_ids, n=20, threshold=3) -> bool:
    """N-gram repetition detection at token id sequence level."""
    L = len(input_ids)
    if L < n:
        return False

    ngrams = [tuple(input_ids[i: i + n]) for i in range(L - n + 1)]
    counts = Counter(ngrams)
    if not counts:
        return False
    return counts.most_common(1)[0][1] >= threshold


def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception:
                continue


def main():
    parser = argparse.ArgumentParser("N-gram repetition filter: print filtered IDs and save to ngram.txt")
    parser.add_argument("--input_jsonl", "-i", default=DEFAULT_INPUT_JSONL, help="Input JSONL path")
    parser.add_argument("--model_path", default=DEFAULT_MODEL_PATH, help="Tokenizer model path")
    parser.add_argument("--field", "-f", default=DEFAULT_FIELD, help="Text field name to check (e.g., solution or code)")
    parser.add_argument("--ngram_n", type=int, default=DEFAULT_NGRAM_N)
    parser.add_argument("--threshold", type=int, default=DEFAULT_THRESHOLD)
    parser.add_argument("--batch_size", type=int, default=DEFAULT_BATCH_SIZE)
    parser.add_argument("--strip_fence", action="store_true",
                        help="If set, extract content inside ```latex ... ``` before tokenization")
    # Key change: default save to ngram.txt
    parser.add_argument("--save_ids", default=DEFAULT_SAVE_IDS,
                        help="Save filtered ids (one per line). Default: ngram.txt")

    args = parser.parse_args()

    tokenizer = load_tokenizer(args.model_path)

    filtered_ids = []
    batch_ids = []
    batch_texts = []

    pbar = tqdm(unit="sample", desc="Scanning")

    def flush_batch():
        nonlocal batch_ids, batch_texts, filtered_ids
        if not batch_ids:
            return

        try:
            enc = tokenizer(batch_texts, add_special_tokens=False, padding=False, verbose=False)
            ids_list = enc["input_ids"]
        except Exception as e:
            print(f"[WARN] Tokenization error, skip batch: {e}")
            batch_ids, batch_texts = [], []
            return

        for sample_id, input_ids in zip(batch_ids, ids_list):
            if has_repetition_ids(input_ids, n=args.ngram_n, threshold=args.threshold):
                filtered_ids.append(sample_id)

        batch_ids, batch_texts = [], []

    for item in iter_jsonl(args.input_jsonl):
        sid = item.get("id", None)
        if sid is None:
            pbar.update(1)
            continue

        txt = item.get(args.field, "")
        if not isinstance(txt, str):
            txt = ""

        if args.strip_fence:
            txt = extract_latex(txt)

        batch_ids.append(sid)
        batch_texts.append(txt)

        if len(batch_ids) >= args.batch_size:
            flush_batch()

        pbar.update(1)

    flush_batch()
    pbar.close()

    # 1) Print to stdout (preserve original behavior)
    for sid in filtered_ids:
        print(sid)

    # 2) Save to ngram.txt (default)
    if args.save_ids:
        save_path = os.path.abspath(args.save_ids)
        save_dir = os.path.dirname(save_path)
        if save_dir and save_dir != ".":
            os.makedirs(save_dir, exist_ok=True)
        with open(save_path, "w", encoding="utf-8") as f:
            for sid in filtered_ids:
                f.write(f"{sid}\n")
        print(f"[OK] Saved {len(filtered_ids)} ids to: {save_path}")

    print("-" * 40)
    print(f"Input: {args.input_jsonl}")
    print(f"Field: {args.field}")
    print(f"N-gram n: {args.ngram_n}, threshold: {args.threshold}")
    print(f"Filtered IDs: {len(filtered_ids)}")
    print("-" * 40)


if __name__ == "__main__":
    main()


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from PIL import Image
import warnings
from tqdm import tqdm

# ================= Configuration =================
JSONL_PATH = Path(os.environ.get("JSONL_PATH", "/path/to/input.jsonl"))
OUT_PATH   = Path(os.environ.get("OUT_PATH", "/path/to/filter.txt"))

THRESH_PIXELS   = 20_000_000
THRESH_MAX_SIDE = 6000
THRESH_AR       = 15.0
MIN_SIDE        = 32

# Ignore PIL's DecompressionBombWarning (we filter by threshold ourselves)
warnings.simplefilter("ignore", Image.DecompressionBombWarning)

# ================= Count total lines first (for progress bar) =================
print("[INFO] counting lines for progress bar...")
with JSONL_PATH.open("r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)
print(f"[INFO] total lines = {total_lines}")

# ================= Scan and filter =================
bad_ids = set()
scanned = 0
bad_json = 0
bad_image = 0

with JSONL_PATH.open("r", encoding="utf-8") as f:
    for line in tqdm(f, total=total_lines, desc="Scan images", ncols=110):
        line = line.strip()
        if not line:
            continue
        scanned += 1
        try:
            obj = json.loads(line)
        except Exception:
            bad_json += 1
            continue

        rid = obj.get("id", None)
        img_path = obj.get("image_path", None)
        if rid is None or not img_path:
            continue

        try:
            rid = int(rid)
        except Exception:
            continue

        try:
            with Image.open(img_path) as im:
                w, h = im.size
        except Exception:
            bad_image += 1
            bad_ids.add(rid)
            continue

        if w == 0 or h == 0:
            bad_ids.add(rid)
            continue

        if min(w, h) < MIN_SIDE:
            bad_ids.add(rid)
            continue

        pixels = w * h
        ar = (w / h) if w >= h else (h / w)

        if (pixels >= THRESH_PIXELS) or (max(w, h) >= THRESH_MAX_SIDE) or (ar >= THRESH_AR):
            bad_ids.add(rid)

# ================= Output =================
bad_ids_sorted = sorted(bad_ids)
OUT_PATH.write_text(
    "\n".join(map(str, bad_ids_sorted)) + ("\n" if bad_ids_sorted else ""),
    encoding="utf-8"
)

print(f"[OK] scanned non-empty lines : {scanned} (bad_json={bad_json})")
print(f"[OK] bad images              : {bad_image} (unopenable images -> filtered)")
print(f"[OK] filtered ids            : {len(bad_ids_sorted)}")
print(f"[OK] saved -> {OUT_PATH}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
r"""
External_Filter.py (Fixed version)
Scan Final_TikZ.parquet to find samples that depend on external files or have severe reference issues.
"""

import os
import re
import pyarrow.parquet as pq

# ========= Configuration =========
PARQUET_PATH = os.environ.get("PARQUET_PATH", "/path/to/input.parquet")
OUT_PATH = os.environ.get("EXTERNAL_FILTER_OUT", "/path/to/external_filter.txt")

BATCH_SIZE = 8192
# =======================


def strip_latex_comments(text: str) -> str:
    """
    Remove LaTeX line comments, while preserving escaped \%
    """
    if not text:
        return ""
    lines = text.splitlines()
    new_lines = []
    for line in lines:
        cut_pos = None
        for i, ch in enumerate(line):
            if ch == "%":
                # Check escape via backslashes
                backslashes = 0
                j = i - 1
                while j >= 0 and line[j] == "\\":
                    backslashes += 1
                    j -= 1
                # Even number of backslashes -> % is not escaped -> comment starts
                if backslashes % 2 == 0:
                    cut_pos = i
                    break
        if cut_pos is not None:
            new_lines.append(line[:cut_pos])
        else:
            new_lines.append(line)
    return "\n".join(new_lines)


def build_external_pattern() -> re.Pattern:
    """
    Build regex: intercept core external dependencies.
    Added plot file and other data reading commands.
    """
    # 1) Simple literals
    simple_literals = [
        # --- Core file includes ---
        r"\input{",
        r"\include{",
        r"\bibliography{",
        r"\addbibresource{",
        r"\subfile{",
        
        # --- [NEW] TikZ/PGFPlots data reading ---
        r"plot file{",       # Intercept \draw plot file{data.txt}
        r"plot file {",      # Intercept with space
        r"\pgfplotstableread{", # Intercept table reading
        r"\pgfplotstabletypeset{", 

        # --- References (external .bib files) ---
        r"\cite{",
        r"\citep{",
        r"\citet{",
        r"\printbibliography",

        # --- Other ---
        r"usepackage{xr}",
    ]
    escaped_literals = [re.escape(s) for s in simple_literals]

    # 2) Special Patterns

    # Intercept non-empty filename in \includegraphics
    # Format: \includegraphics[opt]{filename}
    includegraphics_nonempty = r"\\includegraphics\s*(?:\[[^\]]*])?\s*\{[^}]+\}"

    # [NEW] Intercept \addplot file {...}
    # This is a common PGFPlots external data access
    addplot_file = r"\\addplot\s*(?:\[[^\]]*])?\s*file\s*\{[^}]+\}"

    # 3) Combine all regexes
    patterns = escaped_literals + [includegraphics_nonempty, addplot_file]
    combined = "(" + "|".join(patterns) + ")"

    # Use IGNORECASE to tolerate case variations
    return re.compile(combined, flags=re.MULTILINE | re.IGNORECASE)


EXTERNAL_RE = build_external_pattern()


def safe_str(x) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    try:
        return str(x)
    except Exception:
        return ""


def main():
    if not os.path.exists(PARQUET_PATH):
        raise FileNotFoundError(f"Parquet file not found: {PARQUET_PATH}")

    pf = pq.ParquetFile(PARQUET_PATH)

    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    if os.path.exists(OUT_PATH):
        os.remove(OUT_PATH)

    print(f"Scanning for external dependencies (including plot file): {PARQUET_PATH}")
    print(f"Row groups: {pf.num_row_groups}")
    print(f"Output path: {OUT_PATH}")

    total = 0
    matched = 0

    with open(OUT_PATH, "w", encoding="utf-8") as fout:
        for rg in range(pf.num_row_groups):
            for batch in pf.iter_batches(
                row_groups=[rg],
                batch_size=BATCH_SIZE,
                columns=["id", "code"],
            ):
                id_col = batch["id"]
                code_col = batch["code"]
                n = batch.num_rows

                for i in range(n):
                    total += 1
                    try:
                        _id = str(id_col[i].as_py())
                    except:
                        _id = str(id_col[i]) if id_col[i] is not None else "unknown"

                    code_val = None
                    try:
                        code_val = code_col[i].as_py()
                    except:
                        code_val = str(code_col[i]) if code_col[i] is not None else ""

                    code_str = safe_str(code_val)
                    if not code_str:
                        continue

                    # 1. Remove comments
                    code_no_comments = strip_latex_comments(code_str)

                    # 2. Regex match
                    if EXTERNAL_RE.search(code_no_comments):
                        matched += 1
                        fout.write(_id.strip() + "\n")

            print(
                f"Row group {rg} done. scanned={total}, matched(filtered)={matched}",
                end="\r",
            )

    print("\n\n=== Scan complete ===")
    print(f"Total scanned rows: {total}")
    print(f"Matched external references: {matched}")
    print(f"Result written to: {OUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import AutoTokenizer

# =========================
# Configuration paths
# =========================
JSONL_PATH = Path(os.environ.get("JSONL_PATH", "/path/to/input.jsonl"))
MODEL_PATH = os.environ.get("MODEL_PATH", "/path/to/qwen3-vl-instruct")
FIELD = "solution"
MAX_TOKENS = 8192  # Filter samples with token count > MAX_TOKENS
FILTER_OUT_PATH = os.environ.get("FILTER_OUT_PATH", "filtered_tokens_gt_8192.txt")

assert JSONL_PATH.exists(), f"Not found: {JSONL_PATH}"

# =========================
# Tokenizer
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True, use_fast=True)

# =========================
# Stream + batch tokenize
# =========================
BATCH_SIZE = 512  
token_lens = []
bad_lines = 0
total = 0
filtered_ids = []  # IDs of samples with token count > MAX_TOKENS

batch_ids = []
batch_texts = []

def flush_batch(ids_batch, texts_batch):
    if not texts_batch:
        return
    try:
        enc = tokenizer(texts_batch, add_special_tokens=False, padding=False, verbose=False)
        for sample_id, input_ids in zip(ids_batch, enc["input_ids"]):
            token_len = len(input_ids)
            token_lens.append(token_len)
            # Filter samples with token count > MAX_TOKENS
            if token_len > MAX_TOKENS:
                filtered_ids.append(sample_id)
    except Exception:
        pass

with JSONL_PATH.open("r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Tokenizing", unit="line"):
        line = line.strip()
        if not line:
            continue
        total += 1
        try:
            obj = json.loads(line)
        except Exception:
            bad_lines += 1
            continue

        sample_id = obj.get("id", None)
        s = obj.get(FIELD, "")
        if not isinstance(s, str):
            s = ""
        s = s.strip()

        batch_ids.append(sample_id)
        batch_texts.append(s)
        
        if len(batch_texts) >= BATCH_SIZE:
            try:
                flush_batch(batch_ids, batch_texts)
            except Exception:
                bad_lines += len(batch_texts)
            batch_ids, batch_texts = [], []

# last batch
if batch_texts:
    try:
        flush_batch(batch_ids, batch_texts)
    except Exception:
        bad_lines += len(batch_texts)

token_lens = np.array(token_lens, dtype=np.int32)

# =========================
# Stats
# =========================
def pct(arr, p):
    return int(np.percentile(arr, p)) if arr.size else 0

stats = {
    "total_lines_seen": int(total),
    "count_tokenized": int(token_lens.size),
    "bad_json_or_tokenize_fail": int(bad_lines),
    "filtered_count": len(filtered_ids),  # Samples with token count > MAX_TOKENS
    "min": int(token_lens.min()) if token_lens.size else 0,
    "p50": pct(token_lens, 50),
    "p75": pct(token_lens, 75),
    "p90": pct(token_lens, 90),
    "p95": pct(token_lens, 95),
    "p99": pct(token_lens, 99),
    "max": int(token_lens.max()) if token_lens.size else 0,
    "mean": float(token_lens.mean()) if token_lens.size else 0.0,
}

print(json.dumps(stats, ensure_ascii=False, indent=2))

# Save filtered IDs to file
if filtered_ids:
    filter_path = Path(FILTER_OUT_PATH)
    filter_path.parent.mkdir(parents=True, exist_ok=True)
    with filter_path.open("w", encoding="utf-8") as f:
        for fid in filtered_ids:
            f.write(f"{fid}\n")
    print(f"\n[OK] Saved {len(filtered_ids)} filtered IDs (token count > {MAX_TOKENS}) to: {filter_path}")
else:
    print(f"\n[OK] No samples found with token count > {MAX_TOKENS}")

# =========================
# Plots
# =========================
plt.figure(figsize=(10, 5))
plt.hist(token_lens, bins=120)
plt.xlabel("Token length (solution)")
plt.ylabel("Count")
plt.title("Token length distribution (solution)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(token_lens[token_lens <= 4096], bins=120)
plt.xlabel("Token length (solution) [<=4096]")
plt.ylabel("Count")
plt.title("Token length distribution (solution) (zoom: <=4096)")
plt.tight_layout()
plt.show()
